# MyHumNews — XLM-R on the 19 DVB categories

Fine-tunes `xlm-roberta-base` multi-label on Burmese news from DVB.

**The number to beat is 0.643 macro-F1.** The linear arm (TF-IDF char_wb(3,5) +
one-vs-rest LinearSVC, per-label thresholds) scores 0.658 reading whole
articles, but XLM-R cannot read past 512 tokens. Scored on text cut to those
same 512 tokens, the linear arm gets 0.643 — so 0.643 is the honest target and
the 0.015 gap is the cost of truncation, charged to neither model. Both figures
are in `dvb_xlmr_meta.json` and the notebook reports against both.

The split, the label set and the validation carve-out are baked into
`dvb_xlmr_data.jsonl.gz` as a `split` field, so this notebook cannot drift from
the protocol those numbers were measured under.

**This notebook survives being cut off.** After every epoch it writes both a
results JSON and a full checkpoint (model + optimizer + scheduler), and on
restart it resumes from the last finished epoch. If the session dies at epoch 2
you keep epoch 1's score and continue from there — you never repeat work.
You can deliberately run one epoch per session if runtime is tight.

**Colab:** Runtime -> Change runtime type -> T4 GPU. **Kaggle:** Settings ->
Accelerator -> GPU, attach the data as a Dataset, and set `CKPT_DIR` to
`/kaggle/working` (which persists between sessions of the same notebook).


## 1. Confirm there is a GPU

If this prints nothing, stop and switch the runtime to GPU — on CPU this takes
days, not hours.

In [ ]:
!nvidia-smi || echo 'NO GPU — switch the runtime before going further'

## 2. Dependencies

In [ ]:
!pip -q install -U 'transformers>=4.44' 2>&1 | tail -2
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 3. Where checkpoints go — do this before training

A checkpoint is ~3.3 GB (278M parameters plus AdamW's two moment tensors). It
has to outlive the session, so it belongs on Drive, not in `/content`, which is
wiped when the runtime is recycled.

If you would rather not push 3.3 GB to Drive each epoch, set
`SAVE_OPTIMIZER = False` below: the checkpoint drops to ~1.1 GB and still
resumes, but Adam's moments restart, which costs a little at the resume point.
The results JSON is a few KB and is always saved.

In [ ]:
import os

SAVE_OPTIMIZER = True     # False -> 1.1 GB checkpoints instead of 3.3 GB

CKPT_DIR = None
if os.path.isdir('/kaggle/working'):
    CKPT_DIR = '/kaggle/working'          # persists across sessions on Kaggle
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CKPT_DIR = '/content/drive/MyDrive/myhumnews_xlmr'
    except Exception as e:
        print('Drive not mounted (%s)' % e)
        CKPT_DIR = '/content/myhumnews_xlmr'
        print('!! falling back to %s — THIS IS WIPED when the runtime is '
              'recycled, so a disconnect loses the checkpoint.' % CKPT_DIR)

os.makedirs(CKPT_DIR, exist_ok=True)
CKPT    = os.path.join(CKPT_DIR, 'ckpt.pt')
RESULTS = os.path.join(CKPT_DIR, 'dvb_xlmr_config.json')
print('checkpoint dir:', CKPT_DIR)
print('existing checkpoint:', os.path.exists(CKPT),
      ('%.1f GB' % (os.path.getsize(CKPT) / 1e9)) if os.path.exists(CKPT) else '')

## 4. Locate the data

No upload widget. Put the two files somewhere the notebook can reach once, and
every future session finds them by itself:

- **Colab:** drop `dvb_xlmr_data.jsonl.gz` and `dvb_xlmr_meta.json` into the
  same Drive folder the checkpoints use — `MyDrive/myhumnews_xlmr/`. That is
  the first place this cell looks.
- **Kaggle:** attach them as a Dataset; `/kaggle/input` is searched first.

Anywhere else under `MyDrive` also works — the cell falls back to a recursive
scan. If nothing is found it stops with the paths it tried rather than
half-running.

In [ ]:
import glob

DATA_NAME, META_NAME = 'dvb_xlmr_data.jsonl.gz', 'dvb_xlmr_meta.json'


def locate(name):
    # Cheapest likely locations first; a full Drive scan only as a last resort.
    exact = [
        os.path.join(CKPT_DIR, name),             # alongside the checkpoints
        name,                                     # the working directory
        os.path.join('/content', name),
        os.path.join('/content/drive/MyDrive', name),
        os.path.join('/content/drive/MyDrive/myhumnews_xlmr', name),
        os.path.join('/content/drive/MyDrive/MyHumNews', name),
        os.path.join('/kaggle/working', name),
    ]
    for c in exact:
        if os.path.exists(c):
            return c, 'exact path'

    patterns = [
        '/kaggle/input/**/' + name,               # attached Kaggle datasets
        '/content/drive/MyDrive/*/' + name,        # one folder deep in Drive
        '/content/drive/MyDrive/*/*/' + name,      # two folders deep
    ]
    for pat in patterns:
        hit = glob.glob(pat, recursive=True)
        if hit:
            return hit[0], pat

    print('  scanning all of MyDrive for %s (slow, once)...' % name, flush=True)
    hit = glob.glob('/content/drive/MyDrive/**/' + name, recursive=True)
    if hit:
        return hit[0], 'recursive scan'

    raise FileNotFoundError(
        '%s not found. Tried these exact paths:\n  %s\nand these patterns:\n  %s\n'
        'Put the file in %s (the checkpoint folder) and re-run this cell.'
        % (name, '\n  '.join(exact), '\n  '.join(patterns), CKPT_DIR))


DATA, how_d = locate(DATA_NAME)
META, how_m = locate(META_NAME)
print('data  %s   (%.1f MB, via %s)' % (DATA, os.path.getsize(DATA) / 1e6, how_d))
print('meta  %s   (via %s)' % (META, how_m))

## 5. Load and verify

The counts must read **31855 / 5622 / 24539**. If they do not, the comparison
against 0.658 is void — stop rather than train.

In [ ]:
import gzip, json, numpy as np

meta = json.load(open(META))
LABELS = meta['labels']
L = {l: j for j, l in enumerate(LABELS)}

rows = {'train': [], 'val': [], 'test': []}
with gzip.open(DATA, 'rt', encoding='utf-8') as fh:
    for line in fh:
        d = json.loads(line)
        rows[d['split']].append((d['text'], d['labels']))

def ymat(rs):
    M = np.zeros((len(rs), len(LABELS)), dtype=np.float32)
    for r, (_, ls) in enumerate(rs):
        for l in ls:
            M[r, L[l]] = 1.0
    return M

Xtr = [t for t, _ in rows['train']]; Ytr = ymat(rows['train'])
Xva = [t for t, _ in rows['val']];   Yva = ymat(rows['val'])
Xte = [t for t, _ in rows['test']];  Yte = ymat(rows['test'])

print('train %d  val %d  test %d' % (len(Xtr), len(Xva), len(Xte)))
print('expected ', meta['splits'])
assert [len(Xtr), len(Xva), len(Xte)] == [meta['splits']['train'],
    meta['splits']['val'], meta['splits']['test']], 'SPLIT MISMATCH — stop'
print('labels %d   linear arm to beat: %s'
      % (len(LABELS), meta['linear_arm']['macro_f1_temporal_full_text']))

## 6. Metrics — copied verbatim from `dataset/tune_dvb.py`

Not reimplemented. A subtly different F1 or threshold search is the easiest way
to produce a number that looks comparable to 0.658 and is not.

In [ ]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def best_threshold(scores, gold):
    """Threshold maximising F1 on the validation slice."""
    best_t, best_f = 0.0, -1.0
    for t in np.unique(np.round(scores, 2)):
        pred = scores >= t
        f = prf(int((pred & (gold == 1)).sum()),
                int((pred & (gold == 0)).sum()),
                int((~pred & (gold == 1)).sum()))[2]
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t

## 7. Build the model

One head, 19 sigmoid outputs, BCE — not 19 models; the shared representation is
the reason to expect help on the weak labels (social 0.273, ethnic 0.246 in the
linear arm).

`pos_weight = n_neg/n_pos` from the training split only, capped at 50. This is
the analogue of the linear arm's `class_weight='balanced'`. Without it this task
collapses to all-negatives — that happened for real on the humanitarian arm and
cost five of seven labels.

In [ ]:
import time
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL   = 'xlm-roberta-base'
MAX_LEN = 512      # XLM-R's ceiling; Burmese runs ~586 tokens/article
EPOCHS  = 3
LR      = 2e-5
BATCH   = 16       # 8 on the laptop; a 16 GB GPU takes 16 at 512 with fp16
EVAL_B  = 64
POS_CAP = 50.0
SEED    = meta['seed']

dev = 'cuda'
torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL)

class DS(Dataset):
    def __init__(self, xs, y=None): self.xs, self.y = xs, y
    def __len__(self): return len(self.xs)
    def __getitem__(self, i):
        return self.xs[i], (None if self.y is None else self.y[i])

def collate(batch):
    enc = tok([b[0] for b in batch], truncation=True, max_length=MAX_LEN,
              padding=True, return_tensors='pt')
    if batch[0][1] is None:
        return enc, None
    return enc, torch.tensor(np.stack([b[1] for b in batch]))

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABELS),
    problem_type='multi_label_classification').to(dev)
model.gradient_checkpointing_enable()

n_pos = Ytr.sum(axis=0)
pw = np.minimum(np.where(n_pos > 0, (len(Xtr) - n_pos) / np.maximum(n_pos, 1),
                         1.0), POS_CAP)
print('pos_weight  min %.1f  max %.1f  (capped %.0f)' % (pw.min(), pw.max(), POS_CAP))
lossf = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pw, dtype=torch.float32, device=dev))

dl = DataLoader(DS(Xtr, Ytr), batch_size=BATCH, shuffle=True,
                collate_fn=collate, num_workers=2)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)

bf16 = torch.cuda.is_bf16_supported()
amp_dt = torch.bfloat16 if bf16 else torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=not bf16)
print('autocast', amp_dt, '| steps/epoch', len(dl))

# XLM-R reads at most 512 tokens, so it is scored against the linear arm
# measured on the SAME truncated text, not against its full-text 0.658.
TARGET = meta['linear_arm']['macro_f1_temporal_truncated_512']
FULL   = meta['linear_arm']['macro_f1_temporal_full_text']
print('target %.3f (linear arm @512 tokens); full-text linear %.3f'
      % (TARGET, FULL))

## 8. Resume from the last finished epoch, if there is one

Run this whether or not you think you have a checkpoint. On a fresh start it
does nothing; after a disconnect it restores the model, the optimizer, the LR
schedule position, the per-epoch history and the best-so-far result.

In [ ]:
start_ep, history, best = 1, [], (-1.0, 0, None)
vhistory, best_val = [], -1.0      # the epoch is selected on VAL, not test

if os.path.exists(CKPT):
    st = torch.load(CKPT, map_location=dev, weights_only=False)
    model.load_state_dict(st['model'])
    if st.get('opt') is not None:
        opt.load_state_dict(st['opt'])
        print('restored optimizer state')
    else:
        print('checkpoint has no optimizer state — Adam moments restart')
    sched.load_state_dict(st['sched'])
    if st.get('scaler') is not None:
        scaler.load_state_dict(st['scaler'])
    history  = st['history']
    best     = tuple(st['best'])
    vhistory = st.get('vhistory', [])      # absent in pre-fix checkpoints
    best_val = st.get('best_val', -1.0)
    start_ep = st['epoch'] + 1
    print('resumed after epoch %d; history %s' % (st['epoch'],
          [round(h, 4) for h in history]))
    if history and not vhistory:
        print('!! this checkpoint predates val-based epoch selection, so the\n'
              '   earlier epochs have no val score to compare against and the\n'
              '   final epoch will be selected by default. Every epoch\'s test\n'
              '   score is still listed in macro_f1_by_epoch.')
    if start_ep > EPOCHS:
        print('all %d epochs already done — skip to the final cell' % EPOCHS)
else:
    print('no checkpoint; starting from epoch 1')

## 9. Train

Thresholds are fitted on **val** and scored on **test**. Fitting them on test
would be picking the answer and then grading it.

After each epoch: results JSON written, then the checkpoint written to a temp
path and atomically renamed, so a session killed mid-save cannot corrupt the
previous good checkpoint. Re-running this cell after a disconnect continues
where it stopped.

In [ ]:
def save_results(macro, ep, rec, micro, mp, mr, weighted, n_tot):
    cfg = {'model': MODEL, 'epoch': ep, 'epochs_planned': EPOCHS,
           'max_len': MAX_LEN, 'lr': LR, 'batch': BATCH,
           'pos_weight_cap': POS_CAP,
           'device': torch.cuda.get_device_name(0),
           'macro_f1_temporal': round(macro, 4),
           'micro_f1_temporal': round(micro, 4),
           'weighted_f1_temporal': round(weighted, 4),
           'micro_precision': round(mp, 4), 'micro_recall': round(mr, 4),
           'macro_f1_by_epoch': [round(h, 4) for h in history],
           'val_macro_f1_by_epoch': [round(h, 4) for h in vhistory],
           'epoch_selected_on': 'val',
           'linear_arm_macro_f1_truncated_512': TARGET,
           'linear_arm_macro_f1_full_text': FULL,
           'thresholds': {r[0]: round(r[2], 3) for r in rec},
           'f1': {r[0]: round(r[5], 4) for r in rec}}
    json.dump(cfg, open(RESULTS, 'w'), ensure_ascii=False, indent=1)
    return cfg


def save_ckpt(ep):
    blob = {'model': model.state_dict(),
            'opt': opt.state_dict() if SAVE_OPTIMIZER else None,
            'sched': sched.state_dict(),
            'scaler': scaler.state_dict() if scaler.is_enabled() else None,
            'epoch': ep, 'history': history, 'best': best,
            'vhistory': vhistory, 'best_val': best_val}
    t0 = time.time()
    torch.save(blob, CKPT + '.tmp')
    os.replace(CKPT + '.tmp', CKPT)      # atomic: never corrupts the good one
    print('  checkpoint saved (%.1f GB, %.0fs)'
          % (os.path.getsize(CKPT) / 1e9, time.time() - t0), flush=True)


@torch.no_grad()
def infer(xs):
    model.eval(); out = []
    for enc, _ in DataLoader(DS(xs), batch_size=EVAL_B, shuffle=False,
                             collate_fn=collate, num_workers=2):
        enc = {k: v.to(dev) for k, v in enc.items()}
        with torch.autocast('cuda', dtype=amp_dt):
            out.append(model(**enc).logits.float().cpu().numpy())
    model.train()
    return np.concatenate(out)


sel_agg = sel_agg if 'sel_agg' in dir() else (0.0, 0.0, 0.0, 0.0, 0)

for ep in range(start_ep, EPOCHS + 1):
    model.train(); t0, run = time.time(), 0.0
    for step, (enc, yb) in enumerate(dl, 1):
        enc = {k: v.to(dev) for k, v in enc.items()}
        with torch.autocast('cuda', dtype=amp_dt):
            loss = lossf(model(**enc).logits, yb.to(dev))
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        opt.zero_grad(set_to_none=True)
        run += loss.item()
        if step % 200 == 0:
            el = (time.time() - t0) / 60
            print('  ep%d %d/%d  loss %.4f  %.1fm  eta %.0fm'
                  % (ep, step, len(dl), run / step, el,
                     el / step * (len(dl) - step)), flush=True)

    SV, SB = infer(Xva), infer(Xte)
    f1s, vf1s, rec = [], [], []
    TP = FP = FN = 0          # for micro-F1, which is what MLTC papers report
    for l, j in L.items():
        if Ytr[:, j].sum() == 0 or Yte[:, j].sum() == 0:
            continue
        thr = best_threshold(SV[:, j], Yva[:, j].astype(int))
        # Val score at that threshold. This is what the epoch is SELECTED on;
        # picking the epoch by its test score would be selection on test.
        vp, vg = SV[:, j] >= thr, Yva[:, j] == 1
        vf1s.append(prf(int((vp & vg).sum()), int((vp & ~vg).sum()),
                        int((~vp & vg).sum()))[2])
        pred = SB[:, j] >= thr
        gold = Yte[:, j] == 1
        tp, fp, fn = (int((pred & gold).sum()), int((pred & ~gold).sum()),
                      int((~pred & gold).sum()))
        TP += tp; FP += fp; FN += fn
        p, r, f = prf(tp, fp, fn)
        f1s.append(f); rec.append((l, int(gold.sum()), thr, p, r, f))
    macro  = sum(f1s) / len(f1s)
    vmacro = sum(vf1s) / len(vf1s)
    # Macro weights a 76-example label the same as a 9,426-example one. Micro
    # and weighted are what multi-label papers headline, so all three are
    # reported and the gap between them is itself the long-tail story.
    n_tot = sum(r[1] for r in rec)
    weighted = sum(r[5] * r[1] for r in rec) / n_tot
    mp, mr, micro = prf(TP, FP, FN)
    history.append(macro)
    vhistory.append(vmacro)
    if vmacro > best_val:
        best_val, best = vmacro, (macro, ep, rec)

    print('\n=== epoch %d  test macro-F1 %.3f  (%+.3f vs linear @512)'
          '   val %.3f  [%.0fm] ==='
          % (ep, macro, macro - TARGET, vmacro, (time.time() - t0) / 60))
    print('  micro-F1 %.4f (P %.3f / R %.3f)   weighted-F1 %.4f   over %d '
          'test assignments' % (micro, mp, mr, weighted, n_tot))
    print('  %-34s%6s%8s%7s%7s%7s' % ('label', 'n', 'thr', 'P', 'R', 'F1'))
    for l, n, t, p, r, f in sorted(rec, key=lambda z: -z[5]):
        print('  %-34s%6d%8.2f%7.3f%7.3f%7.3f' % (l, n, t, p, r, f))

    # best-so-far, never lost. The aggregates must describe the epoch that
    # was SELECTED, not whichever ran last, so they are only refreshed when
    # this epoch is the one best_val picked.
    if best[1] == ep:
        sel_agg = (micro, mp, mr, weighted, n_tot)
    save_results(*best, *sel_agg)
    save_ckpt(ep)

print('\n' + '=' * 62)
print('  linear arm, full text                     %.3f' % FULL)
print('  linear arm, first 512 tokens (target)     %.3f' % TARGET)
print(('  XLM-R best (epoch %d, chosen on val)' % best[1]).ljust(44)
      + '%.3f  (%+.3f)' % (best[0], best[0] - TARGET))
print('=' * 62)

## 10. Collect the result

Safe to run in a fresh session without retraining — it reads the JSON the
training cell already wrote to Drive.

In [ ]:
cfg = json.load(open(RESULTS))
print(json.dumps(cfg, ensure_ascii=False, indent=1))
print('\nmacro-F1 by epoch:', cfg['macro_f1_by_epoch'])
tgt = cfg['linear_arm_macro_f1_truncated_512']
print('linear arm @512 %.3f  ->  XLM-R %.3f  (%+.3f)'
      % (tgt, cfg['macro_f1_temporal'], cfg['macro_f1_temporal'] - tgt))
try:
    from google.colab import files
    files.download(RESULTS)
except Exception:
    print('\nnot on Colab — download dvb_xlmr_config.json from the file browser')